In [1]:
import math
import torch
import torch.nn as nn 
import torch.nn.functional as F

In [2]:
torch.manual_seed(42)

In [3]:
d_model=512
num_heads=8
seq_len=6
batch_size=2

In [4]:
encoder_output=torch.randn(batch_size,seq_len,d_model)
decoder_input=torch.randn(batch_size,seq_len,d_model)

In [5]:
print("Encoder Output Shape :", encoder_output.shape)
print("Decoder Input Shape  :", decoder_input.shape)

Encoder Output Shape : torch.Size([2, 6, 512])
Decoder Input Shape  : torch.Size([2, 6, 512])


In [6]:
mask=torch.triu(torch.ones(seq_len,seq_len),diagonal=1)
print(mask)

tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])


In [8]:
mask=mask.masked_fill(mask==1,float("-inf"))
print(mask)

tensor([[0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0.]])


In [10]:
attention=nn.MultiheadAttention(
    embed_dim=d_model,
    num_heads=num_heads,
    batch_first=True
)

In [11]:
masked_output,attention_weights=attention(
    query=decoder_input,
    key=decoder_input,
    value=decoder_input,
    attn_mask=mask
)

In [12]:
print("Masked Output Shape :", masked_output.shape)
print("Attention Shape :", attention_weights.shape)

Masked Output Shape : torch.Size([2, 6, 512])
Attention Shape : torch.Size([2, 6, 6])


In [13]:
cross_attention=nn.MultiheadAttention(
    embed_dim=d_model,
    num_heads=num_heads,
    batch_first=True
)

In [14]:
cross_output,cross_weights=cross_attention(
    query=masked_output,
    key=encoder_output,
    value=encoder_output
)

In [15]:
print("Cross Attention Output Shape :", cross_output.shape)
print("Cross Attention Weights Shape :", cross_weights.shape)

Cross Attention Output Shape : torch.Size([2, 6, 512])
Cross Attention Weights Shape : torch.Size([2, 6, 6])


In [16]:
ffn=nn.Sequential(
    nn.Linear(d_model,d_model*4),
    nn.ReLU(),
    nn.Linear(d_model*4,d_model)
)

In [20]:
ffn_output = ffn(cross_output)

In [21]:
norm1 = nn.LayerNorm(d_model)
norm2 = nn.LayerNorm(d_model)
norm3 = nn.LayerNorm(d_model)

In [22]:
x = norm1(decoder_input + masked_output)

In [23]:
x = norm2(x + cross_output)

In [24]:
decoder_output = norm3(x + ffn_output)

In [25]:
print(decoder_output.shape)

torch.Size([2, 6, 512])
